In [3]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# 1. Load CSV directly from Downloads folder
file_path = os.path.expanduser("~/Downloads/model_features_sampled.csv")
df = pd.read_csv(file_path)

# 2. Features & Target
X = df.drop(columns=["label"])
y = df["label"]

# 3. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. One-Hot Encoding for Categoricals
categorical_cols = ["market", "order_region", "shipping_mode", "category_name"]
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_cols)],
    remainder="passthrough"
)

# 5. Transform Features & Train Model
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

model = LogisticRegression(max_iter=500, C=1.0)
model.fit(X_train_encoded, y_train)

# 6. Evaluation
y_pred = model.predict(X_test_encoded)
y_prob = model.predict_proba(X_test_encoded)[:, 1]

print("=== Model Performance ===")
print(f"Accuracy Score : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"ROC-AUC Score  : {roc_auc_score(y_test, y_prob):.4f}\n")
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


=== Model Performance ===
Accuracy Score : 70.40%
ROC-AUC Score  : 0.7573

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.64      0.84      0.72       926
           1       0.81      0.59      0.68      1074

    accuracy                           0.70      2000
   macro avg       0.72      0.71      0.70      2000
weighted avg       0.73      0.70      0.70      2000



In [5]:
# Extract feature names after One-Hot Encoding
feature_names = preprocessor.get_feature_names_out()

# Create dataframe of feature coefficients
coefficients_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print("=== Top 5 Risk Factors (Increase Late Risk) ===")
print(coefficients_df.head(5).to_string(index=False))

print("\n=== Top 5 Reliable Factors (Decrease Late Risk) ===")
print(coefficients_df.tail(5).to_string(index=False))

=== Top 5 Risk Factors (Increase Late Risk) ===
                          Feature  Coefficient
    cat__order_region_East of USA     1.004713
        cat__category_name_Soccer     0.783788
cat__order_region_Southern Africa     0.758347
          cat__category_name_DVDs     0.738819
         cat__category_name_Music     0.590518

=== Top 5 Reliable Factors (Decrease Late Risk) ===
                              Feature  Coefficient
cat__category_name_Hunting & Shooting    -1.145833
          cat__category_name_Lacrosse    -1.155441
      cat__shipping_mode_Second Class    -2.518962
          cat__shipping_mode_Same Day    -3.613997
    cat__shipping_mode_Standard Class    -4.460940
